# Dense Semantic Search & Two-Stage Neural Retrieval System
### End-to-End Pipeline: Bi-Encoder FAISS Indexing, Cross-Encoder Reranking, BM25 Baseline, and INT8 Quantization

## 1. Setup & Corpus Preparation

In [ ]:
import os
import json
import time
import numpy as np
import torch
import faiss
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

# Load evaluation corpus
with open('../data/sample_corpus.json', 'r', encoding='utf-8') as f:
    corpus = json.load(f)

corpus_texts = [doc['text'] for doc in corpus]
corpus_ids = [doc['id'] for doc in corpus]
print(f"Loaded {len(corpus)} documents into memory.")

## 2. Stage 1: Bi-Encoder Embeddings & FAISS Indexing (<15 ms Retrieval)

In [ ]:
# Load pretrained Bi-Encoder
bi_encoder = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
dim = bi_encoder.get_sentence_embedding_dimension()

# Generate dense normalized representations
doc_embeddings = bi_encoder.encode(corpus_texts, convert_to_numpy=True, normalize_embeddings=True)

# Build FAISS IndexFlatIP (Inner Product with L2-normalized vectors represents Cosine Similarity)
index = faiss.IndexFlatIP(dim)
index.add(doc_embeddings.astype(np.float32))
print(f"FAISS Index successfully built with {index.ntotal} vectors of dimension {dim}.")

## 3. Stage 2: Cross-Encoder Reranker

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

query = "How to accelerate inference with low integer precision?"

# Fast stage 1 retrieval
t0 = time.perf_counter()
q_emb = bi_encoder.encode([query], convert_to_numpy=True, normalize_embeddings=True)
scores, indices = index.search(q_emb.astype(np.float32), k=4)
retrieval_latency = (time.perf_counter() - t0) * 1000
print(f"Stage 1 FAISS Retrieval Latency: {retrieval_latency:.2f} ms\n")

candidate_docs = [corpus[idx] for idx in indices[0]]
print("Stage 1 Top Candidates:")
for doc, s in zip(candidate_docs, scores[0]):
    print(f"[{doc['id']}] (Cosine Score: {s:.4f}): {doc['text']}")

# Stage 2 Cross-Encoder full self-attention rerank
pairs = [[query, doc['text']] for doc in candidate_docs]
rerank_scores = cross_encoder.predict(pairs)
reranked_results = sorted(zip(candidate_docs, rerank_scores), key=lambda x: x[1], reverse=True)

print("\nStage 2 Re-Ranked Results:")
for doc, s in reranked_results:
    print(f"[{doc['id']}] (Cross-Encoder Logit: {s:.4f}): {doc['text']}")

## 4. Evaluation Benchmark: BM25 vs Bi-Encoder vs Two-Stage (MRR@10 & NDCG@10)

In [ ]:
# Tokenize for BM25 baseline
tokenized_corpus = [doc['text'].lower().split() for doc in corpus]
bm25 = BM25Okapi(tokenized_corpus)

test_queries = [
    {"query": "vector similarity search library Meta", "gold_id": "doc_2"},
    {"query": "full self-attention reranking pair", "gold_id": "doc_3"},
    {"query": "probabilistic term matching baseline", "gold_id": "doc_4"},
    {"query": "low latency integer precision acceleration", "gold_id": "doc_5"}
]

def mrr_at_k(gold, retrieved, k=10):
    for i, doc_id in enumerate(retrieved[:k]):
        if doc_id == gold:
            return 1.0 / (i + 1)
    return 0.0

def ndcg_at_k(gold, retrieved, k=10):
    for i, doc_id in enumerate(retrieved[:k]):
        if doc_id == gold:
            return 1.0 / np.log2(i + 2)
    return 0.0

metrics = {
    'BM25 Baseline': {'mrr': [], 'ndcg': []},
    'Bi-Encoder (FAISS)': {'mrr': [], 'ndcg': []},
    'Bi-Encoder + Cross-Encoder': {'mrr': [], 'ndcg': []}
}

for tq in test_queries:
    q, gold = tq['query'], tq['gold_id']
    
    # 1. BM25
    bm25_scores = bm25.get_scores(q.lower().split())
    bm25_ids = [corpus_ids[i] for i in np.argsort(bm25_scores)[::-1]]
    metrics['BM25 Baseline']['mrr'].append(mrr_at_k(gold, bm25_ids))
    metrics['BM25 Baseline']['ndcg'].append(ndcg_at_k(gold, bm25_ids))
    
    # 2. Bi-Encoder
    q_emb = bi_encoder.encode([q], convert_to_numpy=True, normalize_embeddings=True)
    _, hits = index.search(q_emb.astype(np.float32), k=len(corpus))
    bi_ids = [corpus_ids[idx] for idx in hits[0]]
    metrics['Bi-Encoder (FAISS)']['mrr'].append(mrr_at_k(gold, bi_ids))
    metrics['Bi-Encoder (FAISS)']['ndcg'].append(ndcg_at_k(gold, bi_ids))
    
    # 3. Two-Stage Rerank
    candidate_docs = [corpus[idx] for idx in hits[0][:4]]
    pairs = [[q, doc['text']] for doc in candidate_docs]
    re_scores = cross_encoder.predict(pairs)
    reranked = sorted(zip(candidate_docs, re_scores), key=lambda x: x[1], reverse=True)
    reranked_ids = [d[0]['id'] for d in reranked]
    metrics['Bi-Encoder + Cross-Encoder']['mrr'].append(mrr_at_k(gold, reranked_ids))
    metrics['Bi-Encoder + Cross-Encoder']['ndcg'].append(ndcg_at_k(gold, reranked_ids))

print("=== Comparative IR Performance Summary ===")
for method, vals in metrics.items():
    print(f"{method:<30} | MRR@10: {np.mean(vals['mrr']):.4f} | NDCG@10: {np.mean(vals['ndcg']):.4f}")

## 5. INT8 Dynamic Quantization & Latency Profiling

In [ ]:
# Quantize Transformer linear layers to INT8
model_fp32 = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
model_int8 = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2', device='cpu')
model_int8[0].auto_model = torch.ao.quantization.quantize_dynamic(
    model_int8[0].auto_model,
    {torch.nn.Linear},
    dtype=torch.qint8
)

# Warmup
_ = model_fp32.encode(corpus_texts[:2])
_ = model_int8.encode(corpus_texts[:2])

# Latency benchmark
runs = 50
t0 = time.perf_counter()
for _ in range(runs):
    _ = model_fp32.encode(corpus_texts, batch_size=8)
fp32_latency = (time.perf_counter() - t0) / runs * 1000

t0 = time.perf_counter()
for _ in range(runs):
    _ = model_int8.encode(corpus_texts, batch_size=8)
int8_latency = (time.perf_counter() - t0) / runs * 1000

print(f"FP32 Inference Latency: {fp32_latency:.2f} ms")
print(f"INT8 Inference Latency: {int8_latency:.2f} ms")
print(f"Speedup: {fp32_latency / int8_latency:.2f}x")